In [1]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import log_loss, make_scorer
from scipy.stats import spearmanr

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
TRAIN_PATH = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/Train and test/train_split.csv"
TEST_PATH  = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/Train and test/test_split.csv"
OUTPUT_DIR = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs"

TARGET = "pres.abs"
SPECIES_COL = "Species"
NUMERIC_FEATURES = ["easting", "northing", "disturb", "rainann",
                     "soildepth", "soilfert", "tempann", "topo"]
CLIP = 1e-6
N_REPEATS_PERM = 20  # permutation repeats per feature (more = less noisy, slower)

# We use RandomForest as the reference model for this analysis: it handled
# both approaches reasonably well in the main comparison, doesn't need
# feature scaling for splits to make sense, and permutation importance is
# fast and stable for tree ensembles. Swap in another model_name below if
# you'd rather explain a different model's behaviour.
MODEL_NAME = "RandomForest"


def build_reference_model():
    return RandomForestClassifier(
        n_estimators=400, min_samples_leaf=3, class_weight="balanced",
        random_state=42, n_jobs=-1
    )


def neg_log_loss_scorer():
    def _score(estimator, X, y):
        p = np.clip(estimator.predict_proba(X)[:, 1], CLIP, 1 - CLIP)
        return -log_loss(y, p, labels=[0, 1])
    return _score


SCORER = neg_log_loss_scorer()


# ----------------------------------------------------------------------
# SINGLE-SPECIES IMPORTANCE
# ----------------------------------------------------------------------
def single_species_importance(train, test):
    rows = []
    for species, train_sp in train.groupby(SPECIES_COL):
        test_sp = test.loc[test[SPECIES_COL] == species]
        if test_sp.empty or train_sp[TARGET].nunique() < 2:
            continue

        X_train, y_train = train_sp[NUMERIC_FEATURES], train_sp[TARGET].astype(int)
        X_test, y_test = test_sp[NUMERIC_FEATURES], test_sp[TARGET].astype(int)

        pipe = Pipeline([("scale", StandardScaler()), ("clf", build_reference_model())])
        pipe.fit(X_train, y_train)

        result = permutation_importance(
            pipe, X_test, y_test, scoring=SCORER,
            n_repeats=N_REPEATS_PERM, random_state=42
        )
        # importance is defined as the DROP in score when a feature is
        # shuffled, so we negate (result.importances_mean is already
        # "score with permuted feature minus baseline score"; since our
        # scorer is negative log-loss, a bigger positive importance = the
        # feature mattered more)
        for feat, imp in zip(NUMERIC_FEATURES, result.importances_mean):
            rows.append({"species": species, "feature": feat, "importance": imp})

    return pd.DataFrame(rows)


# ----------------------------------------------------------------------
# MULTI-SPECIES (POOLED) IMPORTANCE — evaluated per species subset
# ----------------------------------------------------------------------
def multi_species_importance(train, test):
    X_train = train[NUMERIC_FEATURES + [SPECIES_COL]]
    y_train = train[TARGET].astype(int)

    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), [SPECIES_COL]),
    ])
    pipe = Pipeline([("prep", preprocessor), ("clf", build_reference_model())])
    pipe.fit(X_train, y_train)

    rows = []
    for species, test_sp in test.groupby(SPECIES_COL):
        if test_sp[TARGET].nunique() < 1:
            continue
        X_test_sp = test_sp[NUMERIC_FEATURES + [SPECIES_COL]]
        y_test_sp = test_sp[TARGET].astype(int)

        # Only permute the NUMERIC features here (permuting Species itself
        # isn't meaningful within a single-species subset — it's constant).
        result = permutation_importance(
            pipe, X_test_sp, y_test_sp, scoring=SCORER,
            n_repeats=N_REPEATS_PERM, random_state=42,
            # sklearn permutes whichever columns are in X; restrict to numeric
            # by passing only those columns' positions isn't directly
            # supported pre-1.3, so we permute all and just keep numeric rows
        )
        # result.importances_mean is aligned to columns of X_test_sp, which
        # is NUMERIC_FEATURES + [SPECIES_COL] in that order (see below)
        col_order = NUMERIC_FEATURES + [SPECIES_COL]
        for feat, imp in zip(col_order, result.importances_mean):
            if feat == SPECIES_COL:
                continue
            rows.append({"species": species, "feature": feat, "importance": imp})

    return pd.DataFrame(rows)


def plot_heatmap(df, title, path):
    pivot = df.pivot(index="feature", columns="species", values="importance")
    pivot = pivot.reindex(NUMERIC_FEATURES)  # consistent row order

    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=40, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="Permutation importance (drop in neg-log-loss)")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"[saved] {path}")


def compute_agreement(single_df, multi_df):
    """For each species, Spearman correlation between the single-species
    model's feature ranking and the pooled model's feature ranking for
    that species. High agreement -> pooling should help; low/negative
    agreement -> pooling likely hurts that species."""
    rows = []
    for species in single_df["species"].unique():
        s = single_df[single_df["species"] == species].set_index("feature")["importance"]
        m = multi_df[multi_df["species"] == species].set_index("feature")["importance"]
        common = s.index.intersection(m.index)
        if len(common) < 3:
            continue
        rho, _ = spearmanr(s.loc[common], m.loc[common])
        rows.append({"species": species, "single_vs_pooled_importance_agreement": rho})
    return pd.DataFrame(rows)


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)

    print(f"Computing permutation importance using {MODEL_NAME} as the reference model ...")

    print("  single-species models ...")
    single_imp = single_species_importance(train, test)
    single_path = os.path.join(OUTPUT_DIR, "single_species_importance.csv")
    single_imp.to_csv(single_path, index=False)
    print(f"  [saved] {single_path}")

    print("  pooled multi-species model ...")
    multi_imp = multi_species_importance(train, test)
    multi_path = os.path.join(OUTPUT_DIR, "multi_species_importance.csv")
    multi_imp.to_csv(multi_path, index=False)
    print(f"  [saved] {multi_path}")

    plot_heatmap(single_imp, f"{MODEL_NAME} (single-species): feature importance by species",
                 os.path.join(OUTPUT_DIR, "importance_heatmap_single.png"))
    plot_heatmap(multi_imp, f"{MODEL_NAME} (pooled multi-species): feature importance by species",
                 os.path.join(OUTPUT_DIR, "importance_heatmap_multi.png"))

    agreement = compute_agreement(single_imp, multi_imp)
    agreement_path = os.path.join(OUTPUT_DIR, "species_feature_agreement.csv")
    agreement.to_csv(agreement_path, index=False)
    print(f"[saved] {agreement_path}")

    print("\n===== Feature-ranking agreement (single-species model vs pooled model) =====")
    print(agreement.sort_values("single_vs_pooled_importance_agreement", ascending=False)
          .to_string(index=False))
    print("\nInterpretation: cross-reference this table against the log-loss gain from "
          "pooling (single_vs_multi_species.py -> prevalence_vs_gain.png / per-species "
          "CSVs). Species with LOW agreement here are the ones whose ecology genuinely "
          "differs from the pooled average — that's your explanation for why pooling "
          "hurts them, independent of sample size / prevalence.")


if __name__ == "__main__":
    main()

Computing permutation importance using RandomForest as the reference model ...
  single-species models ...
  [saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs/single_species_importance.csv
  pooled multi-species model ...
  [saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs/multi_species_importance.csv
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs/importance_heatmap_single.png
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs/importance_heatmap_multi.png
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs/species_feature_agreement.csv

===== Feature-ranking agreement (single-species model vs pooled model) =====
                   species  single_vs_pooled_importance_agreement
Coeranoscincus reticulatus                               0.928571
          Cacophis kreftii                               0.8809